# 2-3절 연습 문제 풀이

이 노트북은 2-3절 연습 문제(2-7 ~ 2-10)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `code_examples/ch02/02-03_example.ipynb`를 참고한다.
- 위에서부터 차례대로 실행한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
# 환경 설정 - 시드 고정 (예제 노트북과 같은 SEED)
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [2]:
def train_highlevel(model, X, Y_true, criterion, optimizer, epochs):
    """[코드 2-16]과 같은 고수준 API 학습 루프. 마지막 손실만 반환한다."""
    model.train()
    for _ in range(epochs):
        loss = criterion(model(X), Y_true)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss.item()

## 연습 문제 2-7

> 1장에서 외계 행성의 물리 법칙을 검증하려고 직접 구현한 회귀 분석 모델을, 파이토치 고수준 API로 다시 만들어 보자.

In [3]:
# 1장 1-3절과 같은 데이터를 만든다 (1장 예제의 SEED는 8)
torch.manual_seed(8)
SAMPLE_SIZE = 50
X = torch.rand((SAMPLE_SIZE, 1)) * 10
Y_ideal = 4.9 * X ** 2
Y_true = Y_ideal + torch.randn((SAMPLE_SIZE, 1)) * Y_ideal * 0.1
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]

In [4]:
class QuadraticRegression(nn.Module):
    """y = a * x^2 + b 를 계산하는 회귀 분석 모델"""
    def __init__(self):
        super().__init__()
        # 입력 하나(x^2)를 받아 출력 하나를 내는 선형 계층
        # 가중치가 1장의 a, 편향이 1장의 b에 해당한다
        self.linear_layer = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        return self.linear_layer(x ** 2)      # (40, 1) -> (40, 1)

LR = 0.0001
EPOCHS = 15

torch.manual_seed(SEED)
model = QuadraticRegression()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=LR)
print(model)

QuadraticRegression(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


In [5]:
train_loss = train_highlevel(model, X_train, Y_train, criterion, optimizer, EPOCHS)

model.eval()
with torch.no_grad():
    test_loss = criterion(model(X_test), Y_test).item()

a = model.linear_layer.weight.item()
b = model.linear_layer.bias.item()
print(f'학습 결과: a = {a:.4f}, b = {b:.4f}')
print(f'훈련 손실 = {train_loss:.2f}, 평가 손실 = {test_loss:.2f}')
print('1장 직접 구현 결과: a = 5.12, b = -0.49, 훈련 손실 = 380.41, 평가 손실 = 805.22')

학습 결과: a = 5.1110, b = -0.1529
훈련 손실 = 380.36, 평가 손실 = 801.37
1장 직접 구현 결과: a = 5.12, b = -0.49, 훈련 손실 = 380.41, 평가 손실 = 805.22


### 풀이 해설

핵심은 **y = ax² + b를 선형 계층 하나로 어떻게 표현하느냐**다.
`nn.Linear`는 입력에 가중치를 곱하고 편향을 더하는 계층이므로, 입력으로 x 대신 **x²을 넣으면**
가중치가 a, 편향이 b가 되어 그대로 들어맞는다. 제곱은 `forward()` 안에서 처리했다.

결과는 a = 5.1110, b = -0.1529로 1장에서 직접 구현한 모델(a = 5.12, b = -0.49)과 거의 같고,
훈련 손실도 380.36으로 1장의 380.41과 사실상 같다. 파라미터 초깃값을 정하는 방식이 다를 뿐
**같은 모델을 같은 방법으로 학습하고 있다**는 것을 확인할 수 있다.

직접 구현과 비교하면 파라미터 선언, 기울기 초기화, 최적화 식이 모두 사라졌다.
남은 것은 모델 구조와 학습 루프의 뼈대뿐이다.

### 문제 검토

- **적절성: 적합. 2-3절을 마무리하는 문제로 잘 놓였다.** 1장에서 손으로 짠 것과 2장에서 배운 도구로 만든 것을
  같은 데이터로 비교하게 해, 고수준 API가 '다른 모델'이 아니라 '같은 모델의 간결한 표현'임을 확인시킨다.
  결과 수치까지 거의 일치하므로 독자가 스스로 채점할 수도 있다.
- **[검토] x²을 어디서 처리할지가 숨은 관문이다.** `nn.Linear`는 입력에 가중치를 곱할 뿐이므로,
  제곱을 `forward()` 안에서 할지 데이터를 미리 제곱해 둘지 독자가 스스로 결정해야 한다.
  2장 본문에는 이런 사례가 없어 막히기 쉬운 지점이다. 힌트 한 줄이면 충분하다.
- **[검토] 비교 대상을 지정해 주면 좋다.** '다시 만들어 보자'로 끝나 무엇을 확인해야 할지 없다.
  1장의 결과와 맞춰 보라고 하면 문제의 의도가 분명해진다.

**윤문안**

> **2-7**. 1장에서 외계 행성의 물리 법칙을 검증하려고 직접 구현한 회귀 분석 모델을, 파이토치 고수준 API로
> 다시 만들어 보자. 그리고 학습으로 구한 두 파라미터와 손실을 1장의 결과와 비교해 보자.
>
> 힌트: `nn.Linear`는 입력에 가중치를 곱하고 편향을 더한다. 모델의 입력으로 무엇을 넣으면 y = ax² + b가 되는지 생각해 보자.

## 연습 문제 2-8

> 입력이 세 개인 [연습 문제 2-6]의 AND 게이트 시뮬레이터 모델을 파이토치 고수준 API로 다시 만들어 보자.

In [6]:
X3 = torch.tensor([[0., 0., 0.], [0., 0., 1.], [0., 1., 0.], [0., 1., 1.],
                   [1., 0., 0.], [1., 0., 1.], [1., 1., 0.], [1., 1., 1.]])
Y3 = torch.tensor([[0.], [0.], [0.], [0.], [0.], [0.], [0.], [1.]])

class Perceptron(nn.Module):
    """본문 [코드 2-10]과 같은 구조. 입력 개수만 인자로 받는다."""
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.linear_layer(x)
        return self.activation(x)

def run_and3(epochs, learning_rate, seed=SEED):
    torch.manual_seed(seed)
    model = Perceptron(3, 1)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    loss = train_highlevel(model, X3, Y3, criterion, optimizer, epochs)
    model.eval()
    with torch.no_grad():
        ok = torch.equal((model(X3) >= 0.5).float(), Y3)
    return loss, ok

print(f'{"학습률":>6} {"에포크":>7} {"손실":>9} {"분류 성공":>9}')
print('-' * 38)
for epochs, learning_rate in ((1000, 0.1), (2000, 0.1), (1000, 0.3)):
    loss, ok = run_and3(epochs, learning_rate)
    print(f'{learning_rate:6.1f} {epochs:7d} {loss:9.4f} {str(ok):>9}')

   학습률     에포크        손실     분류 성공
--------------------------------------


   0.1    1000    0.0760     False


   0.1    2000    0.0515      True
   0.3    1000    0.0393      True


### 풀이 해설

[연습 문제 2-6]의 모델을 그대로 옮기면 된다. 직접 만든 가중치 텐서와 편향 텐서가 `nn.Linear(3, 1)` 한 줄로,
직접 만든 `sigmoid()` 함수가 `nn.Sigmoid()` 계층으로, 직접 만든 `optimizer()` 함수가 `optim.SGD`로 바뀐다.
본문 [코드 2-10]의 `Perceptron` 클래스는 입력 개수를 인자로 받으므로 **클래스는 한 글자도 고칠 필요가 없다.**
`Perceptron(2, 1)` 대신 `Perceptron(3, 1)`로 만들기만 하면 된다.

그리고 [연습 문제 2-6]에서 확인한 현상이 여기서도 똑같이 나타난다.
학습률 0.1, 1,000에포크로는 손실이 0.0760까지 내려가지만 (1, 1, 1) 하나를 틀린다.
에포크를 2,000으로 늘리거나 학습률을 0.3으로 올려야 여덟 샘플을 모두 맞힌다.
직접 구현이든 고수준 API든 **학습이 되고 안 되고는 도구가 아니라 하이퍼파라미터가 정한다**는 점을 보여 준다.

### 문제 검토

- **적절성: 적합.** 같은 모델을 두 방식으로 만들어 보게 해 고수준 API가 무엇을 대신해 주는지 또렷하게 드러낸다.
  본문 [코드 2-10]의 `Perceptron` 클래스가 입력 개수를 인자로 받도록 설계된 덕분에 재사용성도 함께 체감된다.
- **[중요] [연습 문제 2-6]과 같은 문제를 그대로 물려받는다.** 2-6이 본문 하이퍼파라미터로 풀리지 않으므로
  이 문제도 그대로 실패한다. 2-6의 지문을 고치면 이 문제도 함께 해결된다.
- **[검토] '다시 만들어 보자'의 확인 기준.** 2-7과 같은 문제다. 직접 구현한 결과와 비교하라는 구절을 넣으면
  두 방식이 같은 결과에 이른다는 점까지 확인하게 된다.

## 연습 문제 2-9

> 0과 1의 조합을 입력받아, 한 출력은 AND 게이트를, 다른 출력은 OR 게이트를 시뮬레이션하는 모델을 만들어 보자.
> 이 모델은 입력 두 개와 출력 두 개를 가진다.

In [7]:
X2 = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
# 첫 번째 열은 AND, 두 번째 열은 OR 게이트의 정답
Y2 = torch.tensor([[0., 0.], [0., 1.], [0., 1.], [1., 1.]])

EPOCHS_MULTI = 3000
torch.manual_seed(SEED)
model = Perceptron(2, 2)          # 출력이 두 개인 선형 계층
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)
loss = train_highlevel(model, X2, Y2, criterion, optimizer, EPOCHS_MULTI)

model.eval()
with torch.no_grad():
    Y_pred = model(X2)
print(f'손실 {loss:.4f}, 분류 성공 {torch.equal((Y_pred >= 0.5).float(), Y2)}')
print(f'\n{"입력":>10} {"AND 출력":>10} {"OR 출력":>9}   정답 (AND, OR)')
for x, y, t in zip(X2.tolist(), Y_pred.tolist(), Y2.tolist()):
    print(f'{str(tuple(x)):>10} {y[0]:10.3f} {y[1]:9.3f}   {tuple(int(v) for v in t)}')
print(f'\n선형 계층의 가중치:\n{model.linear_layer.weight.detach()}')
print(f'편향: {model.linear_layer.bias.detach()}')

손실 0.0341, 분류 성공 True

        입력     AND 출력     OR 출력   정답 (AND, OR)
(0.0, 0.0)      0.029     0.244   (0, 0)
(0.0, 1.0)      0.221     0.856   (0, 1)
(1.0, 0.0)      0.221     0.858   (0, 1)
(1.0, 1.0)      0.728     0.991   (1, 1)

선형 계층의 가중치:
tensor([[2.2472, 2.2447],
        [2.9256, 2.9169]])
편향: tensor([-3.5064, -1.1306])


### 풀이 해설

`nn.Linear(2, 2)`로 출력 개수만 2로 바꾸면 된다. 본문 [그림 2-9]가 설명한 대로 선형 계층의 뉴런 수는
`out_features`가 정하므로, 이 모델은 **같은 두 입력을 공유하는 뉴런 두 개**를 갖는다.
정답 텐서도 `(4, 1)`이 아니라 `(4, 2)` 형태로, 열마다 게이트 하나씩 담으면 된다.

학습된 가중치를 보면 두 뉴런이 서로 다른 값을 가진다. 뉴런마다 독립된 가중치와 편향으로 자기 결정 경계를
따로 학습했기 때문이다. 즉 **하나의 계층 안에서 서로 다른 두 분류기가 동시에 학습된다.**
이것이 3장에서 은닉층의 뉴런 수를 늘리는 일의 의미로 이어진다.

AND 쪽 출력이 OR 쪽보다 0.5에서 가깝게 머무는데, AND 게이트가 (1, 1) 하나만 1로 떼어 내야 해서
[연습 문제 2-6]에서 본 것과 같은 이유로 더 천천히 학습되기 때문이다. 에포크를 1,000으로 줄이면
AND 출력이 0.58에 그쳐 아슬아슬하게 통과한다.

### 문제 검토

- **적절성: 적합. 배치가 좋다.** [그림 2-9]에서 `nn.Linear(2, 3)`의 뉴런 세 개를 설명한 직후라,
  출력 개수를 늘리는 일이 곧 뉴런을 늘리는 일이라는 것을 바로 적용해 보게 한다.
  정답 텐서의 형태를 `(4, 2)`로 바꿔야 한다는 점도 좋은 연습이다.
- **[검토] 두 출력이 독립적으로 학습된다는 점을 확인하게 하면 좋다.** 이 문제의 핵심은 '출력이 두 개'가 아니라
  '뉴런마다 자기 결정 경계를 따로 학습한다'는 것이다. 학습된 가중치를 출력해 비교하라고 하면 그 점이 드러난다.
- **[검토] 에포크 안내.** 본문 예제의 1,000에포크로는 AND 쪽 출력이 0.58로 겨우 문턱을 넘는다.
  결과를 확인해 보고 필요하면 에포크를 늘리라는 구절이 있으면 좋다.

**윤문안**

> **2-9**. 0과 1의 조합을 입력받아, 한 출력은 AND 게이트를, 다른 출력은 OR 게이트를 시뮬레이션하는 모델을
> 만들어 보자. 이 모델은 입력 두 개와 출력 두 개를 가진다.
> 학습을 마친 뒤 선형 계층의 가중치를 출력해, 두 뉴런이 어떤 값을 학습했는지 비교해 보자.

## 연습 문제 2-10

> XOR 게이트는 두 입력이 같으면 0을, 다르면 1을 출력한다. 이 XOR 게이트를 시뮬레이션하는 퍼셉트론 모델을 만들어 보자.
>
> 만약 학습된 퍼셉트론의 분류 결과가 좋지 않다면, 그 이유가 무엇일지 추론해 제시해 보자.
> 참고로 퍼셉트론 모델로 이 데이터를 제대로 분류할 수 없으며, 이를 극복하는 것이 다음 3장의 주요 목표 중 하나다.

In [8]:
Y_xor = torch.tensor([[0.], [1.], [1.], [0.]])    # 두 입력이 다를 때만 1

print(f'{"에포크":>8} {"손실":>9}   네 샘플의 출력')
print('-' * 60)
for epochs in (1000, 5000, 20000):
    torch.manual_seed(SEED)
    model = Perceptron(2, 1)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    loss = train_highlevel(model, X2, Y_xor, criterion, optimizer, epochs)
    model.eval()
    with torch.no_grad():
        out = [round(v, 3) for v in model(X2).flatten().tolist()]
    print(f'{epochs:8d} {loss:9.4f}   {out}')

     에포크        손실   네 샘플의 출력
------------------------------------------------------------


    1000    0.2500   [0.503, 0.499, 0.502, 0.498]


    5000    0.2500   [0.5, 0.5, 0.5, 0.5]


   20000    0.2500   [0.5, 0.5, 0.5, 0.5]


### 풀이 해설

아무리 오래 학습해도 **손실이 0.25에서 더 내려가지 않고, 네 샘플의 출력이 모두 0.5로 수렴한다.**
모델이 입력을 보지 않고 모든 샘플에 같은 값을 내놓는 상태다. 이보다 나은 답을 찾지 못했다는 뜻이다.

이유는 기하학적으로 분명하다. 퍼셉트론의 결정 경계는 직선 하나뿐인데(본문 2-1절),
XOR의 정답은 (0, 1)과 (1, 0)이 1이고 (0, 0)과 (1, 1)이 0이다.
좌표평면에 네 점을 찍어 보면 1인 두 점이 **대각선으로 마주 보고** 있어, 어떤 직선을 그어도 갈라낼 수 없다.
AND, OR, NAND는 모두 한 점만 떼어 내면 되므로 직선으로 가능했지만 XOR은 다르다.

손실이 정확히 0.25인 것도 우연이 아니다. 모든 출력이 0.5일 때 각 샘플의 오차가 ±0.5이고,
제곱하면 0.25, 평균도 0.25다. **모델이 찾을 수 있는 최선이 '아무 정보도 쓰지 않는 것'이었다**는 뜻이다.

이것이 퍼셉트론의 한계이며, 3장에서 은닉층을 넣어 결정 경계를 여러 개 조합하는 것으로 해결한다.

### 문제 검토

- **적절성: 적합. 2장을 닫고 3장을 여는 자리로 매우 잘 설계됐다.** 결과가 명확하게 실패하고(손실 0.25에서 정지,
  출력 전부 0.5), 그 실패가 본문 2-1절의 '직선 하나'라는 설명과 정확히 맞물린다.
  독자가 스스로 한계를 확인한 뒤 3장으로 넘어가게 하는 구성이다.
- **[검토] 마지막 문단의 위치.** "참고로 퍼셉트론 모델로 이 데이터를 제대로 분류할 수 없으며"가 지문에 함께 있어,
  독자가 답을 미리 알고 실험하게 된다. 이 문제의 가치는 스스로 실패를 목격하는 데 있으므로,
  이 한 문장은 뒤로 미루거나 해답 쪽에 두는 편이 낫다고 본다. 다만 독자가 자기 구현을 의심하며 시간을 낭비하지
  않도록 미리 알려 주는 것도 타당한 편집 판단이므로, 판단은 편집자에게 맡긴다.
- **[검토] 관찰 지점을 짚어 주면 좋다.** '분류 결과가 좋지 않다면'보다는, 네 출력이 모두 0.5로 모이고
  손실이 0.25에서 멈춘다는 구체적 현상을 보게 하면 추론의 출발점이 생긴다.

**윤문안**

> **2-10**. XOR 게이트는 두 입력이 같으면 0을, 다르면 1을 출력한다. 이 XOR 게이트를 시뮬레이션하는
> 퍼셉트론 모델을 만들어 보자. 에포크를 크게 늘려도 손실과 네 샘플의 출력이 더 나아지지 않는다면,
> 네 샘플을 좌표평면에 찍어 보고 그 이유가 무엇일지 추론해 제시해 보자.